In [3]:
# DAY 4: SQL DATABASE & BUSINESS QUERIES
# FreshBite Bakery - Inventory Waste Analysis

import sqlite3
import pandas as pd

print("="*60)
print("SQL DATABASE SETUP & ANALYSIS")
print("="*60)

# SECTION 1: CREATE DATABASE & LOAD DATA
print("\n[SECTION 1] Creating Database...")

# Connect to database (creates if doesn't exist)
conn = sqlite3.connect('freshbite_bakery.db')
cursor = conn.cursor()
print("✓ Connected to database")

SQL DATABASE SETUP & ANALYSIS

[SECTION 1] Creating Database...
✓ Connected to database


In [4]:
# Load cleaned CSV files
print("\nLoading cleaned data...")
products = pd.read_csv('products_clean.csv')
sales = pd.read_csv('sales_clean.csv')
inventory = pd.read_csv('inventory_clean.csv')

print(f"✓ Products: {len(products)} rows")
print(f"✓ Sales: {len(sales)} rows")
print(f"✓ Inventory: {len(inventory)} rows")


Loading cleaned data...
✓ Products: 45 rows
✓ Sales: 21151 rows
✓ Inventory: 16437 rows


In [5]:
# Load data into SQLite tables
print("\nLoading tables to database...")
products.to_sql('products', conn, if_exists='replace', index=False)
print("✓ products table created")

sales.to_sql('sales', conn, if_exists='replace', index=False)
print("✓ sales table created")

inventory.to_sql('inventory', conn, if_exists='replace', index=False)
print("✓ inventory table created")


Loading tables to database...
✓ products table created
✓ sales table created
✓ inventory table created


In [6]:
# Create indexes for performance
print("\nCreating indexes...")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_date ON sales(sale_date)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_product ON sales(product_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_inventory_date ON inventory(record_date)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_inventory_product ON inventory(product_id)")
conn.commit()
print("✓ Indexes created")


Creating indexes...
✓ Indexes created


In [7]:
# Verify data loaded
print("\nVerifying data...")
cursor.execute("SELECT COUNT(*) FROM products")
print(f"✓ Products count: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM sales")
print(f"✓ Sales count: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM inventory")
print(f"✓ Inventory count: {cursor.fetchone()[0]}")

print("\n✓ Database ready!")


Verifying data...
✓ Products count: 45
✓ Sales count: 21151
✓ Inventory count: 16437

✓ Database ready!


In [9]:
# SECTION 2: BUSINESS QUERY 1
# Top 10 Products by Spoilage Volume

print("\n" + "="*60)
print("[QUERY 1] Top 10 Products by Spoilage Volume")
print("Business Question: Which products waste the most units?")
print("="*60)

query1 = """
SELECT 
    p.product_name,
    p.category,
    SUM(i.stock_spoiled) AS total_spoiled,
    ROUND(AVG(i.spoilage_rate), 2) AS avg_spoilage_rate_pct
FROM inventory i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.product_name, p.category
ORDER BY total_spoiled DESC
LIMIT 10;
"""

df1 = pd.read_sql_query(query1, conn)
print(df1)
df1.to_csv('query1_top_spoilage_products.csv', index=False)
print("\n✓ Saved: query1_top_spoilage_products.csv")



[QUERY 1] Top 10 Products by Spoilage Volume
Business Question: Which products waste the most units?
           product_name  category  total_spoiled  avg_spoilage_rate_pct
0       Sourdough Bread     Bread         2260.0                    inf
1       Apple Pie Slice       Pie         1219.0                    inf
2     Sandwich - Veggie  Sandwich         1217.0                  16.64
3      Pain au Chocolat    Pastry         1205.0                    inf
4                Eclair    Pastry         1198.0                    inf
5          French Bread     Bread         1190.0                    inf
6         Bagel - Plain     Bagel         1163.0                    inf
7     Carrot Cake Slice      Cake         1163.0                    inf
8         Danish Pastry    Pastry         1156.0                    inf
9  Chocolate Cake Slice      Cake         1154.0                    inf

✓ Saved: query1_top_spoilage_products.csv


In [10]:
# SECTION 3: BUSINESS QUERY 2
# Top 10 Products by Waste Cost

print("\n" + "="*60)
print("[QUERY 2] Top 10 Products by Waste Cost")
print("Business Question: Which products cost us the most money?")
print("="*60)

query2 = """
SELECT 
    p.product_name,
    p.category,
    SUM(i.stock_spoiled) AS total_spoiled,
    p.unit_cost,
    ROUND(SUM(i.stock_spoiled * p.unit_cost), 2) AS total_waste_cost
FROM inventory i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.product_name, p.category, p.unit_cost
ORDER BY total_waste_cost DESC
LIMIT 10;
"""

df2 = pd.read_sql_query(query2, conn)
print(df2)
df2.to_csv('query2_top_waste_cost.csv', index=False)
print("\n✓ Saved: query2_top_waste_cost.csv")


[QUERY 2] Top 10 Products by Waste Cost
Business Question: Which products cost us the most money?
           product_name  category  total_spoiled  unit_cost  total_waste_cost
0     Carrot Cake Slice      Cake         1163.0       4.38           5093.94
1             Rye Bread     Bread         1124.0       4.36           4900.64
2       Sourdough Bread     Bread         1097.0       4.30           4717.10
3         Danish Pastry    Pastry         1156.0       3.96           4577.76
4     Sandwich - Veggie  Sandwich         1217.0       3.64           4429.88
5  Chocolate Cake Slice      Cake         1154.0       3.83           4419.82
6      Almond Croissant    Pastry         1036.0       4.14           4289.04
7           Brioche Bun     Bread         1084.0       3.73           4043.32
8    Pumpernickel Bread     Bread          920.0       4.30           3956.00
9          French Bread     Bread         1190.0       3.24           3855.60

✓ Saved: query2_top_waste_cost.csv


In [11]:
# SECTION 4: BUSINESS QUERY 3
# Spoilage by Store Location

print("\n" + "="*60)
print("[QUERY 3] Spoilage by Store Location")
print("Business Question: Which store has the biggest waste problem?")
print("="*60)

query3 = """
SELECT 
    store_location,
    SUM(stock_spoiled) AS total_spoiled,
    ROUND(AVG(spoilage_rate), 2) AS avg_spoilage_rate_pct,
    COUNT(DISTINCT product_id) AS products_tracked
FROM inventory
GROUP BY store_location
ORDER BY total_spoiled DESC;
"""

df3 = pd.read_sql_query(query3, conn)
print(df3)
df3.to_csv('query3_spoilage_by_store.csv', index=False)
print("\n✓ Saved: query3_spoilage_by_store.csv")


[QUERY 3] Spoilage by Store Location
Business Question: Which store has the biggest waste problem?
  store_location  total_spoiled  avg_spoilage_rate_pct  products_tracked
0       Eastside        16868.0                    inf                45
1       Downtown        16493.0                    inf                45
2       Westside        16369.0                    inf                45

✓ Saved: query3_spoilage_by_store.csv


In [12]:
# SECTION 5: BUSINESS QUERY 4
# Spoilage by Category

print("\n" + "="*60)
print("[QUERY 4] Spoilage by Category")
print("Business Question: Which product category has highest waste?")
print("="*60)

query4 = """
SELECT 
    p.category,
    SUM(i.stock_spoiled) AS total_spoiled,
    ROUND(AVG(i.spoilage_rate), 2) AS avg_spoilage_rate_pct,
    ROUND(SUM(i.stock_spoiled * p.unit_cost), 2) AS total_waste_cost,
    COUNT(DISTINCT p.product_id) AS product_count
FROM inventory i
JOIN products p ON i.product_id = p.product_id
GROUP BY p.category
ORDER BY total_spoiled DESC;
"""

df4 = pd.read_sql_query(query4, conn)
print(df4)
df4.to_csv('query4_spoilage_by_category.csv', index=False)
print("\n✓ Saved: query4_spoilage_by_category.csv")


[QUERY 4] Spoilage by Category
Business Question: Which product category has highest waste?
    category  total_spoiled  avg_spoilage_rate_pct  total_waste_cost  \
0      Bread        17305.0                    inf          42656.52   
1     Pastry         8953.0                    inf          16652.31   
2     Cookie         4432.0                    inf           9336.41   
3   Sandwich         3449.0                    inf           8481.02   
4      Bagel         3403.0                    inf           7034.24   
5        Pie         2366.0                    inf           3472.21   
6       Cake         2317.0                    inf           9513.76   
7    Cupcake         2187.0                    inf           2822.49   
8        Bar         2138.0                    inf           4318.82   
9     Muffin         2122.0                    inf           6626.37   
10    Savory         1058.0                    inf           2623.84   

    product_count  
0              16  
1 

In [13]:
# SECTION 6: BUSINESS QUERY 5
# Monthly Spoilage Trend

print("\n" + "="*60)
print("[QUERY 5] Monthly Spoilage Trend")
print("Business Question: Is waste increasing or decreasing?")
print("="*60)

query5 = """
SELECT 
    strftime('%Y-%m', record_date) AS month,
    SUM(stock_spoiled) AS total_spoiled,
    ROUND(AVG(spoilage_rate), 2) AS avg_spoilage_rate_pct
FROM inventory
GROUP BY month
ORDER BY month;
"""

df5 = pd.read_sql_query(query5, conn)
print(df5)
df5.to_csv('query5_monthly_trend.csv', index=False)
print("\n✓ Saved: query5_monthly_trend.csv")



[QUERY 5] Monthly Spoilage Trend
Business Question: Is waste increasing or decreasing?
     month  total_spoiled  avg_spoilage_rate_pct
0  2024-01         5595.0                    inf
1  2024-02         5039.0                    inf
2  2024-03         5658.0                    inf
3  2024-04         5456.0                    inf
4  2024-05         5517.0                    inf
5  2024-06         5433.0                    inf
6  2024-07         5632.0                    inf
7  2024-08         5743.0                    inf
8  2024-09         5657.0                    inf

✓ Saved: query5_monthly_trend.csv


In [14]:
# SECTION 7: BUSINESS QUERY 6
# Weekend vs Weekday Spoilage

print("\n" + "="*60)
print("[QUERY 6] Weekend vs Weekday Spoilage")
print("Business Question: Does day of week affect spoilage?")
print("="*60)

query6 = """
SELECT 
    CASE 
        WHEN CAST(strftime('%w', record_date) AS INTEGER) IN (0, 6) 
        THEN 'Weekend' 
        ELSE 'Weekday' 
    END AS day_type,
    SUM(stock_spoiled) AS total_spoiled,
    ROUND(AVG(spoilage_rate), 2) AS avg_spoilage_rate_pct,
    COUNT(*) AS record_count
FROM inventory
GROUP BY day_type;
"""

df6 = pd.read_sql_query(query6, conn)
print(df6)
df6.to_csv('query6_weekend_weekday.csv', index=False)
print("\n✓ Saved: query6_weekend_weekday.csv")


[QUERY 6] Weekend vs Weekday Spoilage
Business Question: Does day of week affect spoilage?
  day_type  total_spoiled  avg_spoilage_rate_pct  record_count
0  Weekday        35420.0                    inf         11763
1  Weekend        14310.0                    inf          4674

✓ Saved: query6_weekend_weekday.csv


In [15]:
# SECTION 8: BUSINESS QUERY 7
# High Waste + Low Sales Products

print("\n" + "="*60)
print("[QUERY 7] Products with High Waste + Low Sales")
print("Business Question: Which products should we discontinue?")
print("="*60)

query7 = """
SELECT 
    p.product_name,
    p.category,
    SUM(i.stock_spoiled) AS total_spoiled,
    ROUND(SUM(i.stock_spoiled * p.unit_cost), 2) AS waste_cost,
    COALESCE(SUM(s.quantity_sold), 0) AS total_sold,
    ROUND(SUM(i.stock_spoiled) * 1.0 / NULLIF(SUM(s.quantity_sold), 0), 2) AS waste_to_sales_ratio
FROM inventory i
JOIN products p ON i.product_id = p.product_id
LEFT JOIN sales s ON i.product_id = s.product_id
GROUP BY p.product_name, p.category
HAVING total_spoiled > 100
ORDER BY waste_to_sales_ratio DESC
LIMIT 15;
"""

df7 = pd.read_sql_query(query7, conn)
print(df7)
df7.to_csv('query7_high_waste_low_sales.csv', index=False)
print("\n✓ Saved: query7_high_waste_low_sales.csv")



[QUERY 7] Products with High Waste + Low Sales
Business Question: Which products should we discontinue?
             product_name  category  total_spoiled   waste_cost  total_sold  \
0       Sandwich - Veggie  Sandwich       355364.0   1293524.96      158268   
1       Whole Wheat Bread     Bread       292924.0   1004729.32      131990   
2            French Bread     Bread       336770.0   1091134.80      155660   
3          Bagel - Sesame     Bagel       306816.0    211703.04      147050   
4        Multigrain Bread     Bread       320880.0    725188.80      156520   
5       Gluten Free Bread     Bread       306152.0    303090.48      150768   
6         Vanilla Cupcake   Cupcake       308655.0    379645.65      153768   
7            Banana Bread     Bread       315100.0    182758.00      158040   
8      Red Velvet Cupcake   Cupcake       261648.0    353224.80      131769   
9         Sourdough Bread     Bread      4454922.0  13865212.40     2245056   
10  Chocolate Chip Muffin 

In [16]:
# SECTION 9: BUSINESS QUERY 8
# Store Performance Comparison

print("\n" + "="*60)
print("[QUERY 8] Store Performance Comparison")
print("Business Question: Compare all metrics by store")
print("="*60)

query8 = """
SELECT 
    i.store_location,
    SUM(i.stock_spoiled) AS total_spoiled,
    ROUND(AVG(i.spoilage_rate), 2) AS avg_spoilage_rate,
    COALESCE(SUM(s.quantity_sold), 0) AS total_sold,
    COUNT(DISTINCT i.product_id) AS unique_products,
    ROUND(SUM(i.stock_spoiled * p.unit_cost), 2) AS total_waste_cost
FROM inventory i
JOIN products p ON i.product_id = p.product_id
LEFT JOIN sales s ON i.store_location = s.store_location AND i.product_id = s.product_id
GROUP BY i.store_location
ORDER BY total_waste_cost DESC;
"""

df8 = pd.read_sql_query(query8, conn)
print(df8)
df8.to_csv('query8_store_comparison.csv', index=False)
print("\n✓ Saved: query8_store_comparison.csv")



[QUERY 8] Store Performance Comparison
Business Question: Compare all metrics by store
  store_location  total_spoiled  avg_spoilage_rate  total_sold  \
0       Eastside      2643209.0                inf     1353365   
1       Westside      2526684.0                inf     1328693   
2       Downtown      2580531.0                inf     1361961   

   unique_products  total_waste_cost  
0               45        6302915.78  
1               45        6180066.00  
2               45        6161670.54  

✓ Saved: query8_store_comparison.csv


In [17]:
# CLOSE DATABASE CONNECTION

conn.close()
print("\n" + "="*60)
print("DAY 4 COMPLETE!")
print("="*60)
print("\n📊 Summary:")
print("   • Database created: freshbite_bakery.db")
print("   • 8 business queries executed")
print("   • 8 CSV results saved for visualization")
print("="*60)


DAY 4 COMPLETE!

📊 Summary:
   • Database created: freshbite_bakery.db
   • 8 business queries executed
   • 8 CSV results saved for visualization
